In [1]:
from pyro.infer import Predictive

import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split

from torchvision.datasets import ImageFolder
import pickle

from torch.utils.data import Subset

c:\Users\Revalda Putawara\.conda\envs\bnntest\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import torch

In [5]:
shipsnet_mean = [0.4119, 0.4243, 0.3724]
shipsnet_std = [0.1899, 0.1569, 0.1515]

new_mean = [0.3444, 0.3803, 0.4078]
new_std = [0.0914, 0.0651, 0.0552]


def load_data(batch_size=16):
    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize(mean=shipsnet_mean, 
                             std=shipsnet_std)
    ])

    #dataset = datasets.EuroSAT(root='./data', transform=transform, download=True)
    dataset = ImageFolder(
    root="data/shipsnet/foldered",
    transform=transform
    )
    torch.manual_seed(42)

    #train_size = int(0.8 * len(dataset))
    #test_size = len(dataset) - train_size
    #train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
    
    with open('datasplit/shipsnet_split_indices.pkl', 'rb') as f:
        split = pickle.load(f)
        train_dataset = Subset(dataset, split['train'])
        test_dataset = Subset(dataset, split['test'])

    # Add num_workers and pin_memory for faster data loading
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                             num_workers=4, pin_memory=True, persistent_workers=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size,
                            num_workers=4, pin_memory=True, persistent_workers=True)
    return train_loader, test_loader

In [6]:
train_loader, test_loader = load_data(batch_size=16)

In [7]:
# check  mean and std of the dataset
def check_mean_std(loader):
    mean = 0.
    std = 0.
    nb_samples = 0.

    for data, _ in loader:
        batch_samples = data.size(0)  # batch size (number of samples)
        data = data.view(batch_samples, data.size(1), -1)  # reshape to (batch_size, channels, height*width)
        mean += data.mean(2).sum(0)  # sum over height and width
        std += data.std(2).sum(0)  # sum over height and width
        nb_samples += batch_samples

    mean /= nb_samples
    std /= nb_samples

    return mean, std

In [8]:
check_mean_std(train_loader)

(tensor([-0.0045, -0.0021, -0.0003]), tensor([0.6337, 0.6161, 0.5881]))

In [9]:
check_mean_std(test_loader)

(tensor([0.0193, 0.0084, 0.0012]), tensor([0.6638, 0.6488, 0.6203]))

In [15]:
def load_data(batch_size=16):
    # if shipsnet_std is a torch Tensor, convert to a plain list:
    #scaled_std = [(s / 10.0).item() for s in shipsnet_std]
    scaled_std = [s * 10.0 for s in shipsnet_std]  

    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize(mean=shipsnet_mean,
                             std=scaled_std),
    ])

    dataset = ImageFolder(root="data/shipsnet/foldered",
                          transform=transform)
    torch.manual_seed(42)
    with open('datasplit/shipsnet_split_indices.pkl','rb') as f:
        split = pickle.load(f)
    train_dataset = Subset(dataset, split['train'])
    test_dataset  = Subset(dataset, split['test'])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=4, pin_memory=True, persistent_workers=True)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size,
                              num_workers=4, pin_memory=True, persistent_workers=True)
    return train_loader, test_loader


In [16]:
train_loader, test_loader = load_data(batch_size=16)

In [17]:
check_mean_std(train_loader)

(tensor([-4.5348e-04, -2.1329e-04, -2.7436e-05]),
 tensor([0.0634, 0.0616, 0.0588]))